# Optimization of Algorithms problems

## Exercise 1
### Code Optimization for Text Processing

You are provided with a text processing code to perform the following operations:

1. Convert all text to lowercase.
2. Remove punctuation marks.
3. Count the frequency of each word.
4. Show the 5 most common words.

The code works, but it is inefficient and can be optimized. Your task is to identify areas that can be improved and rewrite those parts to make the code more efficient and readable.

In [1]:
import string

def process_text(text):
    # Text to lowercase
    text = text.lower()

    # Remove punctuation
    for p in string.punctuation:
        text = text.replace(p, "")

    # Split text into words
    words = text.split()

    # Count frecuencies
    frequencies = {}
    for w in words:
        if w in frequencies:
            frequencies[w] += 1
        else:
            frequencies[w] = 1

    sorted_frequencies = sorted(frequencies.items(), key = lambda x: x[1], reverse = True)

    # Get 5 most-common words
    top_5 = sorted_frequencies[:5]
    
    for w, frequency in top_5:
        print(f"'{w}': {frequency} times")

text = """
    In the heart of the city, Emily discovered a quaint little café, hidden away from the bustling streets. 
    The aroma of freshly baked pastries wafted through the air, drawing in passersby. As she sipped on her latte, 
    she noticed an old bookshelf filled with classics, creating a cozy atmosphere that made her lose track of time.
"""
process_text(text)

'the': 5 times
'of': 3 times
'in': 2 times
'a': 2 times
'she': 2 times


Points to optimize:

1. **Removal of punctuation marks**: Using `replace` in a loop can be inefficient, especially with long texts. Look for a more efficient way to remove punctuation marks.
2. **Frequency count**: The code checks for the existence of each word in the dictionary and then updates its count. This can be done more efficiently with certain data structures in Python.
3. **Sort and select:** Consider if there is a more direct or efficient way to get the 5 most frequent words without sorting all the words.
4. **Modularity**: Break the code into smaller functions so that each one performs a specific task. This will not only optimize performance, but also make the code more readable and maintainable.

In [ ]:
# TODO
import io
import string
import timeit
from collections import Counter
from contextlib import redirect_stdout

# Translation table built once: maps every punctuation character to None
PUNCTUATION_TABLE = str.maketrans("", "", string.punctuation)

def clean_text(text):
    """
    Converts the text to lowercase and removes punctuation in a single pass
    (str.translate instead of one str.replace per punctuation mark)
    """
    return text.lower().translate(PUNCTUATION_TABLE)

def count_words(text):
    """
    Counts the frequency of every word (Counter instead of a manual dictionary)
    """
    return Counter(clean_text(text).split())

def top_words(frequencies, n = 5):
    """
    Returns the n most common words (most_common uses a heap, so it does not sort every word)
    """
    return frequencies.most_common(n)

def process_text_optimized(text):
    for w, frequency in top_words(count_words(text)):
        print(f"'{w}': {frequency} times")

# Optimized version
process_text_optimized(text)

# Check: the optimized version must print exactly the same as the original one
with redirect_stdout(io.StringIO()) as original_output:
    process_text(text)
with redirect_stdout(io.StringIO()) as optimized_output:
    process_text_optimized(text)
print(f"\nSame output as the original: {original_output.getvalue() == optimized_output.getvalue()}")

# Time comparison with a much longer text
def run_silently(function, content):
    with redirect_stdout(io.StringIO()):
        function(content)

def compare_times(label, content):
    original_time = timeit.timeit(lambda: run_silently(process_text, content), number = 5)
    optimized_time = timeit.timeit(lambda: run_silently(process_text_optimized, content), number = 5)
    print(f"{label}: original {original_time:.3f} s | optimized {optimized_time:.3f} s | speedup x{original_time / optimized_time:.1f}")

long_text = text * 2000
compare_times("Long text as it is (contains the non-ASCII character 'e with accent')", long_text)
compare_times("Long text with ASCII characters only", long_text.replace("é", "e"))

# Note: Counter is faster than the manual dictionary in both cases, but str.translate
# only has its fast path for pure ASCII strings. With a non-ASCII character in the text
# (like the accent in "cafe") the 32 replace calls of the original can be faster.
# Optimizing means measuring: the best option depends on the data.

## Exercise 2
### Code Optimization for List Processing

You have been given a code that performs operations on a list of numbers for:

1. Filter out even numbers.
2. Duplicate each number.
3. Add all numbers.
4. Check if the result is a prime number.

The code provided achieves its goal, but it may be inefficient. Your task is to identify and improve the parts of the code to increase its efficiency.

In [3]:
import math

def is_prime(n):
    if n <= 1:
        return False
    for i in range(2, int(math.sqrt(n)) + 1):
        if n % i == 0:
            return False
    return True

def process_list(list_):
    filtered_list = []
    for num in list_:
        if num % 2 == 0:
            filtered_list.append(num)
    
    duplicate_list = []
    for num in filtered_list:
        duplicate_list.append(num * 2)
        
    sum = 0
    for num in duplicate_list:
        sum += num

    prime = is_prime(sum)
    
    return sum, prime

list_ = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
result, result_prime = process_list(list_)
print(f"Result: {result}, ¿Prime? {'Yes' if result_prime else 'No'}")

Result: 60, ¿Prime? No


Points to optimize:

1. **Filter numbers**: The code goes through the original list to filter out even numbers. Consider a more efficient way to filter the list.
2. **Duplication**: The list is traversed multiple times. Is there a way to do this more efficiently?
3. **Summing**: The numbers in a list are summed through a loop. Python has built-in functions that can optimize this.
4. **Function `is_prime`**: While this function is relatively efficient, investigate if there are ways to make it even faster.
5. **Modularity**: Consider breaking the code into smaller functions, each focused on a specific task.

In [ ]:
# TODO
import math
import random
import timeit

def filter_even(numbers):
    """
    Yields only the even numbers (lazy generator: no intermediate list is created)
    """
    return (num for num in numbers if num % 2 == 0)

def double(numbers):
    """
    Yields each number multiplied by 2 (lazy generator: no intermediate list is created)
    """
    return (num * 2 for num in numbers)

def is_prime_optimized(n):
    """
    Checks if n is prime: math.isqrt gives an exact integer square root,
    and after discarding the even numbers only odd divisors are tested
    """
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    for i in range(3, math.isqrt(n) + 1, 2):
        if n % i == 0:
            return False
    return True

def process_list_optimized(numbers):
    """
    Filters the even numbers, doubles them, adds them (built-in sum, a single pass)
    and checks if the result is prime
    """
    total = sum(double(filter_even(numbers)))
    return total, is_prime_optimized(total)

# Optimized version
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
result_optimized, result_prime_optimized = process_list_optimized(numbers)
print(f"Result: {result_optimized}, \u00bfPrime? {'Yes' if result_prime_optimized else 'No'}")

# Checks: the optimized version must give exactly the same results as the original one
print(f"\nSame result as the original: {(result_optimized, result_prime_optimized) == process_list(numbers)}")

random.seed(42)
samples = [[random.randint(-50, 50) for _ in range(random.randint(0, 30))] for _ in range(200)]
print(f"Same result on 200 random lists: {all(process_list(s) == process_list_optimized(s) for s in samples)}")
print(f"is_prime_optimized matches is_prime from 0 to 10000: {all(is_prime(n) == is_prime_optimized(n) for n in range(10001))}")

# Time comparison
def compare_times(label, original_function, optimized_function, argument):
    original_time = timeit.timeit(lambda: original_function(argument), number = 3)
    optimized_time = timeit.timeit(lambda: optimized_function(argument), number = 3)
    print(f"{label}: original {original_time:.3f} s | optimized {optimized_time:.3f} s | speedup x{original_time / optimized_time:.1f}")

print()
big_list = list(range(1, 2000001))
compare_times("process_list on 2,000,000 numbers", process_list, process_list_optimized, big_list)
compare_times("is_prime on the prime 999999999989", is_prime, is_prime_optimized, 999999999989)

Both exercises will help you improve your code performance optimization skills and give you a better understanding of how different data structures and programming techniques can affect the efficiency of your code.